# Grid Imaging and Checking

Inspecting images of gridded data is a valuable way of checking for unusual or undesirable artefacts in the data and several functions are provided for this purpose. The main ones are shown here. Some checks are for delivered grids, and it is also possible to generate and image grids from the located data.

This example uses delivered grids from the Canobie Falcon survey and the Vinton Dome FTG survey.

Note that only grids in the `.ERS` format are able to be checked.

___

Import the required python packages, and set the path to the Canobie geowhizz files.

In [ ]:
from pathlib import Path
import galileoQC as qc

In [ ]:
data_root = r'./CanobieData/'
canobieHDF_file = Path(r'./CanobieData/Canobie.hdf5')

___

Ideally, the delivered grids from a survey should cover the same areal extent, have the same grid cell size, the same missing data value, the same geographic datum, and so forth. The **`checkErsHeaders`** function compares this information from the first `.ERS` file it encounters in the given directory with all the others, performing a quick and simple check for consistency.

In [ ]:
qc.checkErsHeaders(Path(data_root))

___

A single `.ERS` grid file can be imaged by the **`display_grid`** function. By default, this produces a shaded image.

In [ ]:
gridfile = Path(data_root + r'FSF_GDD_2p67_final.ERS')
qc.display_grid(gridfile, "Final G_DD", cmap_norm='nonorm', 
    cb_ticks='stats', nSigma=2,
    hs=True, azdeg=45)

___
You can use any Matplotlib colormap, as well as "geosoft" and "parula".

In [ ]:
qc.display_grid(Path(data_root + "FSF_GDD_2p67_final.ERS"), 'Test colormaps', colormap="geosoft")

___

The **`imageAllInDir`** function images all `.ers` grids found in the given directory, using the file name as the plot title. The images use the default color look up table and are shaded. This allows a very quick check that all the grids look okay.

In [ ]:
qc.imageAllInDir(Path(data_root))

___

Finally, we can use **`grid_n_image`** to create grids of any channels in the database and image them for viewing.

Errors in the line number, flight number, fiducial, latitude, or any channel at all are possible and can often be easily seen in an image. Accordingly, **`grid_n_image`** is designed to interpolate every named channel to a regular grid and image the grid.

The grid interpolation can be done by the SciPy `griddata` linear method, or by one of the Verde methods: `neighbours`, `bicubic`, or `biharmonic`, or by `minc` (minimum curvature using the implementation in the python pygmi package). Minimum curvature is the usual choice.

Channels (for example, line number, flight number, day of year) might vary between lines but be constant along a line. Others might vary at a constant rate along a line but change dramatically between lines flown on different days or in different directions. And some might change sign depending on line direction (bearing, velocity, Eotvos correction).

Two pre-filters are provided to deal with these situations: a mean-removal filter subtracts the mean of the channel for each line from the data on that line; and a first difference filter returns the difference between successive samples for the gridding algorithm.

The gridded data are imaged on a linear colour stretch with no shading.

The Canobie located data cover a long and thin area somewhat unsuitable for imaging so we will grid and image the Vinton Dome data instead.

First, let us list the available channels. The default is to grid and image all channels in the database. For a demonstration of the key points above, we only need a few.

In [ ]:
vintonHDF_file = Path(r'./VintonData/VintonDome.hdf5')

In [ ]:
if not vintonHDF_file.exists():
    %run ./Prepare_VintonDomeData.ipynb

In [ ]:
qc.reportChannels(vintonHDF_file, verbose=True)

Here `z_chans` is the list of channels to process. Those in `mr_chans` will have the mean value along each flight-line removed before gridding. Those in `d1_chans` will have the first difference along each flight-line gridded.

The first option is very useful for un-leveled data, and also for channels like heading, or velocity, which are strongly dependent on direction of flight. Here the mean removal filter is applied to `Inline1_raw`. These data are unleveled, and the mean removal acts to level all the data as can be seen by comparing the `Inline1_raw` image with the following on for the `Inline2_raw` channel.

The second option is very useful for data which are expected to vary uniformly with sample number. An example is the `Time` channel in these data. The imaged result shows the result: a completely flat image; any artefacts in the time channel would be very easily seen in this image.

In [ ]:
z_chans = ['Inline1_raw', 'Inline2_raw', 'Tzz_slv', 'Terrain', 'Time', 'Lat']
mr_chans = ['Inline1_raw']
d1_chans = ['Time']
qc.grid_n_image(vintonHDF_file, z_chans, 200.0, mr_chans=mr_chans, d1_chans=d1_chans)

In [ ]:
z_chans = ['Terrain']
qc.grid_n_image(vintonHDF_file, z_chans, 200.0)